# Домашнее задание 7. Сборка конвейера CI/CD

## 1. GitLab pipeline

Был создан [gitlab репозиторий](https://gitlab.com/mishasdk/mipt-ml-service) и код с пайплайном запушен, pipeline был успешно выполнен: https://gitlab.com/mishasdk/mipt-ml-service/-/pipelines/2513783251.

## 2. Обоснование стратегии деплоя

Создадим adr документ с обоснованием выбора Canary стратегии развертывания:

- [0002-canary-deployment-strategy](doc/adr/0002-canary-deployment-strategy.md)

## 3. Реализовать стратегию развертывания

1. Создаем .env файл с конфигурацией и активируем:

In [85]:
!rm -f .env
!cp .env.example .env

1. Запустим сервис локально через docker compose:

In [86]:
!docker compose down
!docker compose up -d

[+] down 0/1
 ⠋ Container mipt-ml-service-nginx-1 Stopping                               0.1s
[+] down 0/1
 ⠙ Container mipt-ml-service-nginx-1 Stopping                               0.2s
[+] down 0/1
 ⠹ Container mipt-ml-service-nginx-1 Removing                               0.3s
[+] down 1/3
 ✔ Container mipt-ml-service-nginx-1  Removed                               0.3s
 ⠋ Container mipt-ml-service-app-v1-1 Stopping                              0.1s
 ⠋ Container mipt-ml-service-app-v2-1 Stopping                              0.1s
[+] down 1/3
 ✔ Container mipt-ml-service-nginx-1  Removed                               0.3s
 ⠙ Container mipt-ml-service-app-v1-1 Stopping                              0.2s
 ⠙ Container mipt-ml-service-app-v2-1 Stopping                              0.2s
[+] down 1/3
 ✔ Container mipt-ml-service-nginx-1  Removed                               0.3s
 ⠹ Container mipt-ml-service-app-v1-1 Stopping                              0.3s
 ⠹ Container mipt-ml-service-ap

2. Для проверки балансировки трафика будем использовать test.sh скрипт:

In [87]:
!bash scripts/test.sh


v0.1.0: 20/20 (100%)
other:  0/20 (0%)


3. Начнем раскатку новой версии:

In [93]:
!bash scripts/canary.sh start
!sleep 3
!bash scripts/test.sh

Canary start: 90% v1 / 10% v2
[+] up 0/1
 ⠋ Container mipt-ml-service-nginx-1 Recreate                               0.1s
[+] up 0/1
 ⠙ Container mipt-ml-service-nginx-1 Recreate                               0.2s
[+] up 0/1
 ⠹ Container mipt-ml-service-nginx-1 Recreate                               0.3s
[+] up 0/1
 ⠸ Container mipt-ml-service-nginx-1 Starting                               0.4s
[+] up 1/1
 ✔ Container mipt-ml-service-nginx-1 Started                                0.4s
Applied: V1=weight=9 | V2=weight=1

v0.1.0: 19/20 (95%)
other:  1/20 (5%)


4. Повышаем долю до 50%:

In [101]:
!bash scripts/canary.sh promote 50

[+] up 0/1
 ⠋ Container mipt-ml-service-nginx-1 Recreate                               0.1s
[+] up 0/1
 ⠙ Container mipt-ml-service-nginx-1 Recreate                               0.2s
[+] up 0/1
 ⠹ Container mipt-ml-service-nginx-1 Recreate                               0.3s
[+] up 0/1
 ⠸ Container mipt-ml-service-nginx-1 Starting                               0.4s
[+] up 1/1
 ✔ Container mipt-ml-service-nginx-1 Started                                0.4s
Applied: V1=weight=5 | V2=weight=5
Traffic promoted to 50% v2


In [103]:
!bash scripts/test.sh


v0.1.0: 10/20 (50%)
other:  10/20 (50%)


5. Если стработал автоматический мониторинг, то можем откатить процесс развертывания:

In [90]:
!bash scripts/canary.sh rollback
!sleep 3
!bash scripts/test.sh

ROLLBACK: 100% traffic to v1
[+] up 0/1
 ⠋ Container mipt-ml-service-nginx-1 Recreate                               0.1s
[+] up 0/1
 ⠙ Container mipt-ml-service-nginx-1 Recreate                               0.2s
[+] up 0/1
 ⠹ Container mipt-ml-service-nginx-1 Recreate                               0.3s
[+] up 0/1
 ⠸ Container mipt-ml-service-nginx-1 Starting                               0.4s
[+] up 1/1
 ✔ Container mipt-ml-service-nginx-1 Started                                0.4s
Applied: V1=weight=1 | V2=down

v0.1.0: 20/20 (100%)
other:  0/20 (0%)


6. Полная раскатка, весь трафик направляется через v2 сервис:

In [91]:
!bash scripts/canary.sh promote 100
!sleep 3
!bash scripts/test.sh

[+] up 0/1
 ⠋ Container mipt-ml-service-nginx-1 Recreate                               0.1s
[+] up 0/1
 ⠙ Container mipt-ml-service-nginx-1 Recreate                               0.2s
[+] up 0/1
 ⠹ Container mipt-ml-service-nginx-1 Recreate                               0.3s
[+] up 1/1
 ✔ Container mipt-ml-service-nginx-1 Started                                0.4s
Applied: V1=backup | V2=weight=1
Traffic promoted to 100% v2

v0.1.0: 0/20 (0%)
other:  20/20 (100%)


## 4. Спланировать A/B-тестирование для ML-модели

Предположим, у нас есть задача классификации и две модели — старая (контроль) и новая (тест).

#### Гипотезы

- **H_0:** новая модель не лучше старой по целевой метрике
- **H_1:** новая модель улучшает целевую метрику на ≥ MDE (минимальный детектируемый эффект)

#### Метрики

| Тип | Метрика | Описание |
|-----|---------|----------|
| Целевая (бизнес) | Конверсия | Доля пользователей, совершивших целевое действие |
| Прокси (модели) | Accuracy | Доля корректных предсказаний |
| Guardrail (защитные) | Latency p95 / p99 | Время отклика — не должно деградировать |

#### Разбиение пользователей

Пользователи разбиваются на бакеты по `user_id`, один пользователь всегда попадает в один бакет.

Разбиение трафика: 50 / 50.

#### MDE — Minimum Detectable Effect

Минимальный эффект, который мы хотим уметь детектировать:

- По accuracy: **+1 п.п.**
- По конверсии: **+2%** (например, 10% → 10.2%)

## 5. Развертывание сервиса с помощью GitHub Actions 

Был добавлен конфиг для развертывания сервиса после пуша коммита в main: [deploy.yml](.github/workflows/deploy.yml).

Развертывание просходит после каждого коммита в Yandex Cloud в виде serverless приложения. 

Проверка работоспособности сервиса:

In [92]:
!curl https://bbals8fqd5ljtvbml3g7.containers.yandexcloud.net/health

{"status":"ok","version":"1.0.7"}